# Phase 8D - URL-Only Feature Engineering

This notebook converts the legitimate UCI raw URL dataset into a clean URL-only feature table. It uses only deterministic URL-string measurements. UCI fields derived from HTML, page content, DNS, reputation, search, or traffic are not used.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.url_dataset import engineer_url_dataset
from src.url_feature_extraction import URL_ONLY_FEATURE_ORDER

raw = pd.read_csv(ROOT / 'dataset' / 'raw' / 'PhiUSIIL_Phishing_URL_Dataset.csv')
engineered, quality = engineer_url_dataset(raw)
output = ROOT / 'dataset' / 'processed' / 'url_only_features.csv'
output.parent.mkdir(parents=True, exist_ok=True)
engineered.to_csv(output, index=False)
print('Quality:', quality)
print('Feature count:', len(URL_ONLY_FEATURE_ORDER))
print('Features:', URL_ONLY_FEATURE_ORDER)
print('Class distribution:')
display(engineered['label'].value_counts().sort_index().rename('count').to_frame())
print('Domain groups:', engineered['domain_group'].nunique())
assert set(engineered['label'].unique()) == {0, 1}
assert engineered[URL_ONLY_FEATURE_ORDER].isna().sum().sum() == 0
(ROOT / 'results' / 'url_feature_engineering_summary.json').write_text(json.dumps({**quality, 'feature_count': len(URL_ONLY_FEATURE_ORDER), 'features': URL_ONLY_FEATURE_ORDER, 'class_distribution': {str(k): int(v) for k, v in engineered['label'].value_counts().sort_index().items()}}, indent=2), encoding='utf-8')
print('Saved:', output)

Quality: {'invalid_rows_removed': 10, 'duplicate_urls_removed': 425, 'valid_rows': 235360, 'unique_domains': 167154}
Feature count: 27
Features: ['url_length', 'hostname_length', 'path_length', 'query_length', 'fragment_length', 'dot_count', 'hyphen_count', 'underscore_count', 'digit_count', 'special_character_count', 'slash_count', 'query_parameter_count', 'url_depth', 'subdomain_count', 'uses_https', 'uses_ip_address', 'contains_at_symbol', 'encoded_character_count', 'suspicious_character_count', 'url_entropy', 'hostname_entropy', 'path_entropy', 'letter_count', 'numeric_count', 'digit_ratio', 'special_character_ratio', 'looks_like_shortener']
Class distribution:


,count
label,
0,100510
1,134850


Domain groups: 167154
Saved: ./dataset\processed\url_only_features.csv


## Excluded UCI fields

The engineered file contains `URL`, URL-only numeric features, `label`, and `domain_group` for splitting. It excludes `Title`, `LineOfCode`, `TLDLegitimateProb`, `URLTitleMatchScore`, page/resource counts, redirects, robots, and other fields that require webpage or external information.